# Notebook 1 · Análisis temporal del consumo y la facturación — Electro Puno S.A.A.

**Objetivo.** Entender cómo evolucionan mes a mes el consumo (kWh) y la facturación (S/), detectar cambios importantes y evaluar —con cautela— si hay un patrón estacional.

**Preguntas que responde**
1. ¿Cómo evoluciona el consumo? 2. ¿Y la facturación? 3. ¿Qué meses son máximos y mínimos? 4. ¿Hay patrón estacional? 5. ¿Cuándo hay aumentos o caídas importantes? 6. ¿Qué tan comparables son los años?

**Cómo usarlo.** Ejecuta las celdas de arriba hacia abajo (`Run All`). Es independiente de los otros notebooks. Los resultados pesados se guardan en `analisis/resultados/` y se reutilizan si los datos no cambiaron.

**Requisitos:** `pip install duckdb pandas numpy plotly scipy statsmodels jinja2 nbformat ipython`

> **Regla de oro de este notebook:** *un mes sin datos no es un mes con consumo cero.* Los meses faltantes se muestran como huecos, nunca como ceros, y las variaciones que "saltan" un hueco no se calculan.

In [24]:
# ============================================================
# CONFIGURACIÓN COMÚN (idéntica en los tres notebooks)
# Qué hace: importa librerías, localiza la raíz del proyecto,
# comprueba que existan los archivos y abre DuckDB.
# ============================================================
import sys, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

def verificar_paquetes(requeridos):
    """requeridos: {nombre_del_módulo: nombre_en_pip}"""
    import importlib
    faltan = []
    for modulo, paquete in requeridos.items():
        try:
            importlib.import_module(modulo)
        except ImportError:
            faltan.append(paquete)
    if faltan:
        raise ImportError(
            "Faltan paquetes en el entorno de este kernel: " + ", ".join(faltan)
            + "\nInstálalos con:\n    pip install " + " ".join(faltan)
            + "\ny reinicia el kernel."
        )

verificar_paquetes({'duckdb': 'duckdb', 'pandas': 'pandas', 'numpy': 'numpy', 'plotly': 'plotly', 'scipy': 'scipy'})

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
try:
    from statsmodels.tsa.seasonal import STL
    HAY_STATSMODELS = True
except ImportError:
    HAY_STATSMODELS = False
    print('⚠️ statsmodels no está instalado: se omitirá la descomposición STL (sección opcional).')

pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---------- 1) Localizar la raíz del proyecto (robusto en VS Code) ----------
def encontrar_raiz():
    """Busca hacia arriba (y un nivel hacia abajo) la carpeta que contiene
    preprocesamiento/silver. Sirve aunque el directorio de trabajo cambie."""
    inicios = [Path.cwd().resolve()]
    ruta_nb = globals().get("__vsc_ipynb_file__")      # VS Code define esta variable
    if ruta_nb:
        inicios.insert(0, Path(ruta_nb).resolve().parent)
    for inicio in inicios:
        for base in [inicio, *inicio.parents]:
            if (base / "preprocesamiento" / "silver").is_dir():
                return base
        for hijo in inicio.glob("*/preprocesamiento/silver"):
            return hijo.parent.parent
    raise FileNotFoundError(
        "No encuentro la carpeta 'preprocesamiento/silver'.\n"
        f"Directorio de trabajo actual: {Path.cwd()}\n"
        "Abre VS Code en la carpeta 'visualizacion-datos-trabajo' (o dentro de ella) "
        "y vuelve a ejecutar."
    )

RAIZ = encontrar_raiz()
SILVER = RAIZ / "preprocesamiento" / "silver"
RESULTADOS = RAIZ / "analisis" / "resultados"      # resultados intermedios (no son los datos originales)
RESULTADOS.mkdir(parents=True, exist_ok=True)

ARCHIVOS = {
    "clientes":   SILVER / "consumo_clientes.parquet",
    "agregado":   SILVER / "consumo_agregado.parquet",
    "resumen":    SILVER / "resumen_mensual.parquet",
    "rechazados": SILVER / "_rechazados.parquet",
}
OPCIONALES = {"rechazados"}
for nombre, ruta in ARCHIVOS.items():
    if not ruta.exists():
        if nombre in OPCIONALES:
            print(f" Archivo opcional no encontrado: {ruta.name}")
        else:
            raise FileNotFoundError(f"Falta el archivo obligatorio: {ruta}")

# ---------- 2) Conexión DuckDB y vistas (no se carga nada en memoria) ----------
con = duckdb.connect()
con.execute("SET preserve_insertion_order = false")   # menos memoria en agregaciones grandes

def _lit(ruta):                                       # escapa comillas simples para SQL
    return str(ruta.as_posix()).replace("'", "''")

for nombre, ruta in ARCHIVOS.items():
    if ruta.exists():
        con.execute(f"CREATE OR REPLACE VIEW {nombre} AS SELECT * FROM read_parquet('{_lit(ruta)}')")

def q(sql):
    """Ejecuta SQL en DuckDB y devuelve un DataFrame de pandas."""
    return con.execute(sql).df()

def sql_lista(valores):
    """Convierte una lista de textos en ('A','B') seguro para SQL."""
    if len(valores) == 0:
        return "(NULL)"
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

# ---------- 3) Validación de columnas ----------
COLUMNAS_OBLIGATORIAS = {
    "clientes": ["codigo_cliente","ubigeo","departamento","provincia","distrito","tarifa",
                 "periodo","fecha_alta","consumo_kwh","facturacion_soles"],
    "agregado": ["ubigeo","departamento","provincia","distrito","tarifa","periodo",
                 "consumo_kwh","facturacion_soles","clientes"],
    "resumen":  ["periodo","anio","mes","granularidad","n_suministros",
                 "consumo_kwh_total","facturacion_soles_total","consumo_kwh_promedio"],
}
for nombre, cols in COLUMNAS_OBLIGATORIAS.items():
    existentes = set(q(f"DESCRIBE SELECT * FROM {nombre}")["column_name"])
    faltan = [c for c in cols if c not in existentes]
    if faltan:
        raise KeyError(f"En '{ARCHIVOS[nombre].name}' faltan las columnas: {faltan}. "
                       f"Columnas encontradas: {sorted(existentes)}")

# ---------- 4) Huella de los datos (para reutilizar resultados guardados) ----------
HUELLA = hashlib.md5("|".join(
    f"{p.name}:{p.stat().st_size}:{p.stat().st_mtime_ns}" for p in ARCHIVOS.values() if p.exists()
).encode()).hexdigest()[:8]

def cachear(nombre, sql):
    """Si ya existe el resultado guardado con la misma huella de datos, lo lee;
    si no, ejecuta la consulta y lo guarda en analisis/resultados/ (Parquet)."""
    ruta = RESULTADOS / f"nb01_{nombre}_{HUELLA}.parquet"
    if ruta.exists():
        return con.execute(f"SELECT * FROM read_parquet('{_lit(ruta)}')").df()
    for viejo in RESULTADOS.glob(f"nb01_{nombre}_*.parquet"):
        viejo.unlink()                                 # elimina versiones obsoletas
    df = q(sql)
    con.register("_tmp_cache", df)
    con.execute(f"COPY _tmp_cache TO '{_lit(ruta)}' (FORMAT PARQUET)")
    con.unregister("_tmp_cache")
    return df

# ---------- 5) Constantes y utilidades compartidas ----------
SEMILLA = 42                                           # semilla para todo muestreo
MESES = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
COLOR_DETALLE  = "#1f77b4"    # azul: 2023-2025 (detalle por suministro)
COLOR_AGREGADO = "#ff7f0e"    # naranja: 2026 (agregado)
COLOR_CONSUMO  = "#2ca02c"    # verde: consumo (kWh)
COLOR_FACTURA  = "#d62728"    # rojo: facturación (S/)
# Fecha de alta <= este valor se trata como "dudosa" (valor por defecto/centinela probable).
FECHA_ALTA_DUDOSA_HASTA = "1989-01-01"

def fmt_n(x, d=0):
    return "s/d" if pd.isna(x) else f"{x:,.{d}f}"

def fmt_pct(x, d=1):
    return "s/d" if pd.isna(x) else f"{x:+.{d}f}%"

print(f"Raíz del proyecto : {RAIZ}")
print(f"Carpeta silver    : {SILVER}")
print(f"Resultados        : {RESULTADOS}")
print(f"Huella de datos   : {HUELLA}")
print("Archivos: " + ", ".join(f"{n} ({p.stat().st_size/1e6:,.1f} MB)" for n, p in ARCHIVOS.items() if p.exists()))

Raíz del proyecto : D:\PROYECTO\visualizacion-datos-trabajo
Carpeta silver    : D:\PROYECTO\visualizacion-datos-trabajo\preprocesamiento\silver
Resultados        : D:\PROYECTO\visualizacion-datos-trabajo\analisis\resultados
Huella de datos   : 222a6326
Archivos: clientes (100.1 MB), agregado (0.6 MB), resumen (0.0 MB), rechazados (0.0 MB)


## 1. ¿Qué contiene cada archivo y qué análisis permite?

Antes de calcular, revisamos **esquemas, filas y periodos**. Esto evita comparar cosas que no son comparables.

* `consumo_clientes` → una fila = un suministro en un mes (2023–2025). Permite detalle y distribuciones.
* `consumo_agregado` → 2026. Cada fila es un **grupo** de suministros (`clientes` = cuántos). Sirve para totales y conteos, **no** para distribuciones individuales.
* `resumen_mensual` → un registro por mes (totales). Es el punto de partida de este notebook.

In [5]:
# --- Esquemas y tamaño de cada archivo ---
filas = {n: con.execute(f"SELECT COUNT(*) FROM {n}").fetchone()[0] for n in ARCHIVOS if ARCHIVOS[n].exists()}
resumen_archivos = []
for n in filas:
    esq = q(f"DESCRIBE SELECT * FROM {n}")
    resumen_archivos.append({"archivo": ARCHIVOS[n].name, "filas": filas[n],
                             "columnas": len(esq), "lista de columnas": ", ".join(esq["column_name"])})
display(pd.DataFrame(resumen_archivos).style.format({"filas": "{:,.0f}"}))

if filas.get("rechazados", 0) == 0:
    print("rechazados.parquet está vacío: el preprocesamiento no rechazó filas.")

# --- Periodos disponibles por archivo ---
p_cli = q("SELECT MIN(periodo) AS desde, MAX(periodo) AS hasta, COUNT(DISTINCT periodo) AS n_periodos FROM clientes")
p_agr = q("SELECT MIN(periodo) AS desde, MAX(periodo) AS hasta, COUNT(DISTINCT periodo) AS n_periodos FROM agregado")
print("\nconsumo_clientes :", p_cli.iloc[0].to_dict())
print("consumo_agregado :", p_agr.iloc[0].to_dict())

# --- ¿Existen nulos en las columnas que usaremos? ---
nulos = q("""SELECT COUNT(*) AS filas,
       SUM(CASE WHEN consumo_kwh IS NULL THEN 1 ELSE 0 END) AS consumo_nulo,
       SUM(CASE WHEN facturacion_soles IS NULL THEN 1 ELSE 0 END) AS facturacion_nula,
       SUM(CASE WHEN consumo_kwh < 0 THEN 1 ELSE 0 END) AS consumo_negativo,
       SUM(CASE WHEN facturacion_soles < 0 THEN 1 ELSE 0 END) AS facturacion_negativa
FROM clientes""")
print("\nCalidad básica en consumo_clientes:")
display(nulos)

,archivo,filas,columnas,lista de columnas
0,consumo_clientes.parquet,"11,432,753",15,"codigo_cliente, ubigeo, departamento, provincia, distrito, tarifa, estado_cliente, periodo, fecha_alta, consumo_kwh, facturacion_soles, fecha_corte, clientes, granularidad, source_file"
1,consumo_agregado.parquet,"74,485",15,"codigo_cliente, ubigeo, departamento, provincia, distrito, tarifa, estado_cliente, periodo, fecha_alta, consumo_kwh, facturacion_soles, fecha_corte, clientes, granularidad, source_file"
2,resumen_mensual.parquet,41,8,"periodo, anio, mes, granularidad, n_suministros, consumo_kwh_total, facturacion_soles_total, consumo_kwh_promedio"
3,_rechazados.parquet,0,15,"codigo_cliente, ubigeo, departamento, provincia, distrito, tarifa, estado_cliente, periodo, fecha_alta, consumo_kwh, facturacion_soles, fecha_corte, clientes, granularidad, source_file"


rechazados.parquet está vacío: el preprocesamiento no rechazó filas.

consumo_clientes : {'desde': 202301, 'hasta': 202512, 'n_periodos': 33}
consumo_agregado : {'desde': 202601, 'hasta': 202608, 'n_periodos': 8}

Calidad básica en consumo_clientes:


,filas,consumo_nulo,facturacion_nula,consumo_negativo,facturacion_negativa
0,11432753,0.00,0.00,0.00,"18,767.00"


## 2. Cobertura: ¿qué meses existen realmente?

Construimos un calendario completo desde el primer hasta el último mes y marcamos qué meses tienen datos. Esto responde parte de la **pregunta 6** (comparabilidad) y es imprescindible antes de hablar de tendencias.

**Indicador:** estado de cada mes = *detalle* (suministro a suministro), *agregado* (2026) o *sin datos*.
**Gráfico:** mapa año × mes. **Cómo leerlo:** las casillas grises son meses que **no** podemos analizar.

In [6]:
resumen = q("SELECT * FROM resumen ORDER BY periodo")
resumen["fecha"] = pd.to_datetime(resumen["periodo"].astype(str) + "01", format="%Y%m%d")

calendario = pd.date_range(resumen["fecha"].min(), resumen["fecha"].max(), freq="MS")

# m = tabla mensual sobre el calendario COMPLETO (los meses faltantes quedan como NaN, no como 0)
m = (resumen.set_index("fecha")
            .reindex(calendario)
            .rename(columns={"consumo_kwh_total": "kwh", "facturacion_soles_total": "fact",
                             "n_suministros": "n", "consumo_kwh_promedio": "kwh_promedio_resumen"}))
m.index.name = "fecha"
m["anio"] = m.index.year
m["mes"]  = m.index.month
m["estado"] = np.where(m["kwh"].isna(), "sin datos",
               np.where(m["granularidad"] == "agregado", "agregado", "detalle"))

faltantes = m.index[m["estado"] == "sin datos"]
print(f"Meses en el calendario: {len(m)} | con datos: {(m['estado']!='sin datos').sum()} | sin datos: {len(faltantes)}")
print("Meses SIN datos:", [f.strftime("%Y-%m") for f in faltantes] or "ninguno")

# --- Mapa de cobertura ---
codigo = m["estado"].map({"sin datos": 0, "detalle": 1, "agregado": 2})
piv_z = pd.DataFrame({"anio": m["anio"], "mes": m["mes"], "z": codigo}).pivot(index="anio", columns="mes", values="z")
piv_t = pd.DataFrame({"anio": m["anio"], "mes": m["mes"], "t": m["estado"]}).pivot(index="anio", columns="mes", values="t")
piv_z = piv_z.reindex(columns=range(1, 13)); piv_t = piv_t.reindex(columns=range(1, 13))

fig = go.Figure(go.Heatmap(
    z=piv_z.values, x=MESES, y=[str(a) for a in piv_z.index],
    text=piv_t.values, texttemplate="%{text}", hovertemplate="%{y}-%{x}: %{text}<extra></extra>",
    colorscale=[[0, "#d9d9d9"], [0.333, "#d9d9d9"], [0.334, COLOR_DETALLE], [0.666, COLOR_DETALLE],
                [0.667, COLOR_AGREGADO], [1, COLOR_AGREGADO]],
    zmin=0, zmax=2, showscale=False, xgap=3, ygap=3))
fig.update_layout(title="Cobertura mensual de los datos (gris = sin datos, azul = detalle, naranja = agregado)",
                  xaxis_title="Mes", yaxis_title="Año", yaxis_autorange="reversed", height=300)
fig.show()

Meses en el calendario: 44 | con datos: 41 | sin datos: 3
Meses SIN datos: ['2024-06', '2024-12', '2025-02']


### 2.1 Validación cruzada: ¿el resumen coincide con los archivos detallados?

Antes de confiar en `resumen_mensual`, recalculamos los totales con DuckDB desde los archivos detallados y comparamos.
En 2026 el número de suministros es `SUM(clientes)` (no el número de filas, porque cada fila es un grupo).

In [7]:
det = q("""SELECT periodo, COUNT(*)::BIGINT AS n_det, SUM(consumo_kwh) AS kwh_det, SUM(facturacion_soles) AS fact_det
           FROM clientes GROUP BY 1""")
agr = q("""SELECT periodo, SUM(clientes)::BIGINT AS n_det, SUM(consumo_kwh) AS kwh_det, SUM(facturacion_soles) AS fact_det
           FROM agregado GROUP BY 1""")
val = pd.concat([det, agr]).merge(
    resumen[["periodo", "n_suministros", "consumo_kwh_total", "facturacion_soles_total"]], on="periodo", how="outer")
val["dif_n"]    = val["n_det"] - val["n_suministros"]
val["dif_kwh_%"]  = 100 * (val["kwh_det"]  / val["consumo_kwh_total"] - 1)
val["dif_fact_%"] = 100 * (val["fact_det"] / val["facturacion_soles_total"] - 1)
mx = val[["dif_n", "dif_kwh_%", "dif_fact_%"]].abs().max()
print("Diferencia máxima absoluta entre resumen y recálculo:")
print(mx.to_string())
if (mx["dif_n"] > 0) or (mx["dif_kwh_%"] > 0.001) or (mx["dif_fact_%"] > 0.001):
    print("⚠️ Hay diferencias: revisar qué periodos no coinciden.")
    display(val[(val["dif_n"].abs() > 0) | (val["dif_kwh_%"].abs() > 0.001) | (val["dif_fact_%"].abs() > 0.001)])
else:
    print("El resumen mensual coincide con los archivos detallados (suministros, kWh y soles).")

Diferencia máxima absoluta entre resumen y recálculo:
dif_n        0.00
dif_kwh_%    0.00
dif_fact_%   0.00
El resumen mensual coincide con los archivos detallados (suministros, kWh y soles).


### 2.2 ¿Son comparables 2023–2025 (detalle) y 2026 (agregado)?

Los **totales** (kWh, S/) y el **número de suministros** sí pueden compararse si ambos archivos miden lo mismo. No lo damos por hecho: probamos si el salto entre **dic‑2025 → ene‑2026** es "normal" comparado con los demás cambios mensuales. Si fuera un salto extraordinario, sospecharíamos un cambio de definición.

Lo que **no** es comparable entre ambas etapas son las *distribuciones individuales* (medianas, percentiles): en 2026 no hay suministros individuales.

In [8]:
m["kwh_ps"]  = m["kwh"] / m["n"]            # kWh por suministro-mes
m["fact_ps"] = m["fact"] / m["n"]           # S/ por suministro-mes
m["ratio"]   = m["fact"] / m["kwh"]         # S/ por kWh (razón económica, ver notebook 3)

def cambio_robusto(serie):
    """Cambio logarítmico entre meses CONSECUTIVOS (NaN si falta alguno de los dos)."""
    return np.log(serie).diff()

filas_comp = []
for col, nombre in [("n", "Suministros"), ("kwh", "Consumo total"), ("kwh_ps", "kWh por suministro"), ("fact", "Facturación total")]:
    d = cambio_robusto(m[col]).dropna()
    salto = d.get(pd.Timestamp("2026-01-01"), np.nan)
    mediana = d.median(); mad = 1.4826 * (d - mediana).abs().median()
    z = (salto - mediana) / mad if mad > 0 else np.nan
    filas_comp.append({"indicador": nombre, "cambio dic‑25→ene‑26 (%)": 100 * (np.exp(salto) - 1) if pd.notna(salto) else np.nan,
                       "cambio mensual típico (mediana, %)": 100 * (np.exp(mediana) - 1),
                       "puntaje robusto del salto": z})
comp = pd.DataFrame(filas_comp)
display(comp.style.format({"cambio dic‑25→ene‑26 (%)": "{:+.2f}", "cambio mensual típico (mediana, %)": "{:+.2f}", "puntaje robusto del salto": "{:+.1f}"}))
print("Lectura: un puntaje robusto con valor absoluto > 3 indicaría que el salto de dic‑25 a ene‑26 es inusual frente a los demás cambios.")

,indicador,cambio dic‑25→ene‑26 (%),"cambio mensual típico (mediana, %)",puntaje robusto del salto
0,Suministros,+0.09,+0.17,-0.6
1,Consumo total,+2.22,+0.08,+0.7
2,kWh por suministro,+2.13,-0.01,+0.7
3,Facturación total,+0.77,+0.62,+0.0


Lectura: un puntaje robusto con valor absoluto > 3 indicaría que el salto de dic‑25 a ene‑26 es inusual frente a los demás cambios.


## 3. Indicadores mensuales

Construimos, sobre el calendario completo:
* **Variación mensual (MoM):** solo entre meses **consecutivos**. Si el mes anterior no existe, queda vacía (no se "salta" el hueco).
* **Variación interanual (YoY):** contra el **mismo mes del año anterior**, solo si ambos existen.
* **Por suministro:** dividir entre `n_suministros` quita el efecto de que la cartera crece, y permite ver si el *comportamiento* cambia.

In [9]:
for col in ["kwh", "fact", "kwh_ps", "fact_ps", "n", "ratio"]:
    m[f"{col}_mom"] = m[col].pct_change(fill_method=None) * 100       # vs mes anterior inmediato
    m[f"{col}_yoy"] = (m[col] / m[col].shift(12) - 1) * 100            # vs mismo mes del año anterior

# ---- Estadísticos descriptivos (sobre meses con datos) ----
def descr(serie):
    s = serie.dropna()
    return pd.Series({"meses": s.size, "media": s.mean(), "mediana": s.median(), "desv. estándar": s.std(),
                      "coef. variación (%)": 100 * s.std() / s.mean(), "mínimo": s.min(), "máximo": s.max()})

for etiqueta, mask in [("DETALLE 2023–2025", m["estado"] == "detalle"),
                       ("AGREGADO 2026", m["estado"] == "agregado"),
                       ("TODOS LOS MESES CON DATOS", m["estado"] != "sin datos")]:
    t = pd.DataFrame({"Consumo total (kWh)": descr(m.loc[mask, "kwh"]),
                      "Facturación total (S/)": descr(m.loc[mask, "fact"]),
                      "kWh por suministro": descr(m.loc[mask, "kwh_ps"]),
                      "S/ por suministro": descr(m.loc[mask, "fact_ps"])})
    print(f"\n=== {etiqueta} ===")
    display(t.style.format("{:,.2f}"))


=== DETALLE 2023–2025 ===


,Consumo total (kWh),Facturación total (S/),kWh por suministro,S/ por suministro
meses,33.00,33.00,33.00,33.00
media,"21,191,311.95","28,172,322.43",61.14,81.23
mediana,"21,341,976.76","28,164,875.40",61.67,81.11
desv. estándar,"1,177,527.08","2,830,441.65",2.67,7.07
coef. variación (%),5.56,10.05,4.36,8.71
mínimo,"17,806,276.02","21,899,917.40",52.84,64.99
máximo,"23,113,599.53","32,498,674.00",65.08,91.93



=== AGREGADO 2026 ===


,Consumo total (kWh),Facturación total (S/),kWh por suministro,S/ por suministro
meses,8.00,8.00,8.00,8.00
media,"23,578,390.35","30,980,210.04",65.20,85.65
mediana,"23,711,268.02","31,209,330.15",65.43,86.93
desv. estándar,"1,285,415.18","3,096,059.56",2.88,7.79
coef. variación (%),5.45,9.99,4.41,9.10
mínimo,"21,441,124.48","25,977,438.60",59.98,72.66
máximo,"25,139,145.88","34,968,837.50",68.89,95.33



=== TODOS LOS MESES CON DATOS ===


,Consumo total (kWh),Facturación total (S/),kWh por suministro,S/ por suministro
meses,41.00,41.00,41.00,41.00
media,"21,657,083.34","28,720,202.94",61.93,82.09
mediana,"21,551,889.00","28,956,627.00",61.81,82.77
desv. estándar,"1,521,732.32","3,058,719.74",3.13,7.33
coef. variación (%),7.03,10.65,5.05,8.93
mínimo,"17,806,276.02","21,899,917.40",52.84,64.99
máximo,"25,139,145.88","34,968,837.50",68.89,95.33


### 3.1 Meses máximos y mínimos (pregunta 3)

Se muestran por **total** y por **suministro**. Como la cartera crece con el tiempo, los máximos del total tienden a caer en los años recientes; la versión por suministro ayuda a separar "más clientes" de "cada cliente consume más".

In [10]:
datos = m[m["estado"] != "sin datos"]
def top(col, n=3):
    a = datos.nlargest(n, col)[[col]].assign(tipo="máximo")
    b = datos.nsmallest(n, col)[[col]].assign(tipo="mínimo")
    t = pd.concat([a, b]); t.index = t.index.strftime("%Y-%m"); t.index.name = "mes"
    return t
extremos = {"kwh": "Consumo total (kWh)", "fact": "Facturación total (S/)",
            "kwh_ps": "kWh por suministro", "fact_ps": "S/ por suministro"}
for col, nombre in extremos.items():
    print(f"\n{nombre}")
    display(top(col).style.format({col: "{:,.2f}"}))


Consumo total (kWh)


,kwh,tipo
mes,,
2026-06,"25,139,145.88",máximo
2026-08,"24,772,823.63",máximo
2026-07,"24,646,740.79",máximo
2023-03,"17,806,276.02",mínimo
2023-02,"18,285,801.85",mínimo
2023-01,"19,738,537.44",mínimo



Facturación total (S/)


,fact,tipo
mes,,
2026-08,"34,968,837.50",máximo
2026-06,"34,289,339.10",máximo
2025-06,"32,498,674.00",máximo
2023-03,"21,899,917.40",mínimo
2023-02,"23,283,496.40",mínimo
2023-05,"24,075,024.50",mínimo



kWh por suministro


,kwh_ps,tipo
mes,,
2026-06,68.89,máximo
2026-08,67.53,máximo
2026-07,67.36,máximo
2023-03,52.84,mínimo
2023-02,54.30,mínimo
2025-03,56.69,mínimo



S/ por suministro


,fact_ps,tipo
mes,,
2026-08,95.33,máximo
2026-06,93.97,máximo
2025-06,91.93,máximo
2023-03,64.99,mínimo
2023-02,69.13,mínimo
2023-05,71.13,mínimo


## 4. Evolución del consumo y la facturación (preguntas 1 y 2)

**Qué gráfico:** serie mensual. Azul = detalle 2023–2025; naranja = agregado 2026; franjas grises = meses sin datos; la línea punteada une ambas etapas.
**Cómo leerlo:** pase el cursor para ver total, suministros y variaciones. Use el zoom (arrastrar) para acercarse.
**Limitación:** un hueco significa *no hay archivo*, no que se consumió cero.

In [11]:
def marca_vertical(fig, fecha, texto):
    fig.add_shape(type="line", x0=fecha, x1=fecha, y0=0, y1=1, xref="x", yref="paper",
                  line=dict(color="gray", dash="dot", width=1))
    fig.add_annotation(x=fecha, y=1.02, xref="x", yref="paper", text=texto, showarrow=False, font=dict(size=10, color="gray"))

def sombrear_faltantes(fig):
    for f in faltantes:
        fig.add_vrect(x0=f - pd.Timedelta(days=15), x1=f + pd.Timedelta(days=15),
                      fillcolor="lightgray", opacity=0.45, line_width=0, layer="below")
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="Mes sin datos",
                             marker=dict(symbol="square", size=12, color="lightgray")))

def texto_hover(col, unidad, d=0):
    out = []
    for f, r in m.iterrows():
        if r["estado"] == "sin datos":
            out.append(f"{f:%Y-%m}<br>SIN DATOS"); continue
        out.append(f"<b>{f:%Y-%m}</b> ({r['estado']})<br>{unidad}: {fmt_n(r[col], d)}"
                   f"<br>Suministros: {fmt_n(r['n'])}"
                   f"<br>Var. mensual: {fmt_pct(r[col + '_mom'])}<br>Var. interanual: {fmt_pct(r[col + '_yoy'])}")
    return out

def serie_mensual(col, titulo, ytitulo, unidad, d=0):
    hov = texto_hover(col, unidad, d)
    fig = go.Figure()
    for estado, color, nombre in [("detalle", COLOR_DETALLE, "Detalle 2023–2025"), ("agregado", COLOR_AGREGADO, "Agregado 2026")]:
        y = m[col].where(m["estado"] == estado)
        fig.add_trace(go.Scatter(x=m.index, y=y, mode="lines+markers", name=nombre, line=dict(color=color, width=2),
                                 marker=dict(size=7), hovertext=hov, hoverinfo="text", connectgaps=False))
    ult = m.index[m["estado"] == "detalle"].max(); pri = m.index[m["estado"] == "agregado"].min()
    if pd.notna(ult) and pd.notna(pri):
        fig.add_trace(go.Scatter(x=[ult, pri], y=[m.loc[ult, col], m.loc[pri, col]], mode="lines", showlegend=False,
                                 line=dict(color="gray", dash="dot", width=1), hoverinfo="skip"))
        marca_vertical(fig, pri, "inicio agregado")
    sombrear_faltantes(fig)
    fig.update_layout(title=titulo, xaxis_title="Mes", yaxis_title=ytitulo, hovermode="closest",
                      legend=dict(orientation="h", y=-0.2), height=470)
    fig.update_yaxes(tickformat=",.0f")
    return fig

serie_mensual("kwh", "Consumo eléctrico mensual total", "Consumo (kWh)", "Consumo kWh").show()

In [12]:
serie_mensual("fact", "Facturación mensual total", "Facturación (S/)", "Facturación S/").show()

**Interpretación guiada.** Compare la forma de ambas curvas: si la facturación sube o baja *sin* que el consumo lo haga, el motivo no es el volumen de energía (puede ser precio, estructura tarifaria, cargos fijos o regularizaciones). La sección 7 profundiza en eso con la razón S/kWh.

## 5. Comparación por mes calendario entre años (pregunta 4)

**Qué se analiza:** si un mismo mes (p. ej. junio) se comporta parecido cada año.
**Gráfico:** una línea por año; izquierda = total; derecha = por suministro (quita el efecto del crecimiento de cartera).
**Limitación:** hay solo 3 años completos como máximo y algunos meses faltan; una coincidencia visual no prueba estacionalidad.

In [13]:
colores_anio = {2023: "#1f77b4", 2024: "#2ca02c", 2025: "#d62728", 2026: COLOR_AGREGADO}
fig = make_subplots(rows=1, cols=2, subplot_titles=("Consumo total (kWh)", "Consumo por suministro (kWh/suministro)"), horizontal_spacing=0.1)
for anio, g in m.groupby("anio"):
    g = g.reindex(pd.date_range(f"{anio}-01-01", periods=12, freq="MS"))   # 12 meses; faltantes = NaN
    for j, col in enumerate(["kwh", "kwh_ps"], start=1):
        fig.add_trace(go.Scatter(x=MESES, y=g[col].values, mode="lines+markers", name=str(anio),
                                 legendgroup=str(anio), showlegend=(j == 1), line=dict(color=colores_anio.get(anio), dash="dash" if anio == 2026 else "solid"),
                                 hovertemplate=f"{anio} %{{x}}<br>%{{y:,.2f}}<extra></extra>", connectgaps=False), row=1, col=j)
fig.update_xaxes(title_text="Mes calendario")
fig.update_yaxes(title_text="kWh", row=1, col=1, tickformat=",.0f")
fig.update_yaxes(title_text="kWh por suministro", row=1, col=2)
fig.update_layout(title="Consumo por mes calendario y año (línea discontinua = 2026, agregado)", height=460, legend_title="Año")
fig.show()

## 6. Variaciones importantes (pregunta 5)

**Indicador:** variación porcentual mensual (MoM) y variación interanual (YoY).
**Criterio para "importante":** usamos un umbral estadístico robusto —más de **2.5 desviaciones robustas (MAD)** respecto a la mediana de los cambios— para no depender de un porcentaje arbitrario. Las marcas ✖ señalan esos meses.
**Cómo leerlo:** barras ausentes = no hay mes anterior consecutivo (hueco de datos).

In [14]:
def marcar_inusuales(serie_mom, k=2.5):
    s = serie_mom.dropna(); med = s.median(); mad = 1.4826 * (s - med).abs().median()
    return s[(s - med).abs() > k * mad], med, mad

inus_kwh, med_k, mad_k = marcar_inusuales(m["kwh_mom"])
inus_fac, med_f, mad_f = marcar_inusuales(m["fact_mom"])

fig = go.Figure()
fig.add_trace(go.Bar(x=m.index, y=m["kwh_mom"], name="Consumo (kWh)", marker_color=COLOR_CONSUMO,
                     hovertemplate="%{x|%Y-%m}<br>Consumo: %{y:+.2f}%<extra></extra>"))
fig.add_trace(go.Bar(x=m.index, y=m["fact_mom"], name="Facturación (S/)", marker_color=COLOR_FACTURA,
                     hovertemplate="%{x|%Y-%m}<br>Facturación: %{y:+.2f}%<extra></extra>"))
fig.add_trace(go.Scatter(x=inus_kwh.index, y=inus_kwh.values, mode="markers", name="Cambio inusual (consumo)",
                         marker=dict(symbol="x", size=11, color="black"), hoverinfo="skip"))
fig.add_trace(go.Scatter(x=inus_fac.index, y=inus_fac.values, mode="markers", name="Cambio inusual (facturación)",
                         marker=dict(symbol="x", size=11, color="purple"), hoverinfo="skip"))
fig.update_layout(title="Variación porcentual mensual vs. mes anterior consecutivo", xaxis_title="Mes",
                  yaxis_title="Variación mensual (%)", barmode="group", legend=dict(orientation="h", y=-0.25), height=470)
fig.show()

print(f"Cambio mensual típico del consumo: mediana {med_k:+.2f}% | umbral inusual ±{2.5*mad_k:.2f} pp alrededor de la mediana")
print(f"Cambio mensual típico de la facturación: mediana {med_f:+.2f}% | umbral inusual ±{2.5*mad_f:.2f} pp alrededor de la mediana")
print("Meses inusuales en consumo     :", {f.strftime('%Y-%m'): round(v, 1) for f, v in inus_kwh.items()})
print("Meses inusuales en facturación :", {f.strftime('%Y-%m'): round(v, 1) for f, v in inus_fac.items()})

Cambio mensual típico del consumo: mediana +0.08% | umbral inusual ±7.57 pp alrededor de la mediana
Cambio mensual típico de la facturación: mediana +0.62% | umbral inusual ±22.58 pp alrededor de la mediana
Meses inusuales en consumo     : {'2023-04': 15.4, '2025-04': 8.5, '2026-04': 11.2}
Meses inusuales en facturación : {}


### 6.1 Variación interanual (YoY)

Compara cada mes con **el mismo mes del año anterior**, lo que elimina la estacionalidad "de calendario". Solo aparecen meses donde ambos existen. Para 2026 vs. 2025 la comparación es de **totales y suministros** (compatibles), no de distribuciones.

In [15]:
yoy = m[["kwh_yoy", "fact_yoy", "kwh_ps_yoy", "n_yoy"]].dropna(how="all")
yoy = yoy[yoy.notna().any(axis=1)]
print(f"Meses con comparación interanual disponible: {yoy['kwh_yoy'].notna().sum()}")

fig = go.Figure()
for col, nombre, color in [("kwh_yoy", "Consumo total", COLOR_CONSUMO), ("fact_yoy", "Facturación total", COLOR_FACTURA),
                           ("kwh_ps_yoy", "kWh por suministro", "#17becf")]:
    fig.add_trace(go.Bar(x=m.index, y=m[col], name=nombre, marker_color=color,
                         hovertemplate="%{x|%Y-%m}<br>" + nombre + ": %{y:+.2f}% vs. mismo mes año anterior<extra></extra>"))
marca_vertical(fig, pd.Timestamp("2026-01-01"), "2026 (agregado)")
fig.update_layout(title="Variación interanual (mismo mes del año anterior)", xaxis_title="Mes", yaxis_title="Variación interanual (%)",
                  barmode="group", legend=dict(orientation="h", y=-0.25), height=450)
fig.show()

res_yoy = (m.assign(anio_obs=m["anio"])[["anio_obs", "kwh_yoy", "fact_yoy", "kwh_ps_yoy"]].dropna(how="all", subset=["kwh_yoy", "fact_yoy"])
             .groupby("anio_obs").agg(meses=("kwh_yoy", "count"), kwh_yoy_mediana=("kwh_yoy", "median"),
                                      fact_yoy_mediana=("fact_yoy", "median"), kwh_ps_yoy_mediana=("kwh_ps_yoy", "median")))
print("\nMediana de la variación interanual por año observado (%):")
display(res_yoy.style.format("{:,.2f}"))

Meses con comparación interanual disponible: 26



Mediana de la variación interanual por año observado (%):


,meses,kwh_yoy_mediana,fact_yoy_mediana,kwh_ps_yoy_mediana
anio_obs,,,,
2024,10.00,3.51,18.35,1.30
2025,9.00,3.55,4.57,1.51
2026,7.00,9.03,1.65,5.80


## 7. Comparación consumo vs. facturación con selector de variable

Use el **menú desplegable** (dentro del gráfico) para cambiar la variable. La opción *"Índice base 100"* pone consumo y facturación en la misma escala (100 = promedio de 2023, el único año completo) para ver si se mueven juntos.

In [16]:
base23 = m.loc["2023-01-01":"2023-12-01"]
variables = {
    "Consumo total (kWh)":        ("kwh", "kWh", ",.0f"),
    "Facturación total (S/)":     ("fact", "S/", ",.0f"),
    "Consumo por suministro (kWh)": ("kwh_ps", "kWh/suministro", ",.2f"),
    "Facturación por suministro (S/)": ("fact_ps", "S/ por suministro", ",.2f"),
    "Razón facturación/consumo (S/ por kWh)": ("ratio", "S/ por kWh", ",.3f"),
    "Número de suministros": ("n", "suministros", ",.0f"),
}
fig = go.Figure(); grupos = []
for nombre, (col, unidad, fmt) in variables.items():
    idx = []
    for estado, color, etq in [("detalle", COLOR_DETALLE, "Detalle"), ("agregado", COLOR_AGREGADO, "Agregado 2026")]:
        fig.add_trace(go.Scatter(x=m.index, y=m[col].where(m["estado"] == estado), mode="lines+markers", name=etq,
                                 line=dict(color=color), visible=(len(grupos) == 0), connectgaps=False,
                                 hovertemplate="%{x|%Y-%m}<br>%{y:" + fmt + "} " + unidad + "<extra></extra>"))
        idx.append(len(fig.data) - 1)
    grupos.append((nombre, idx, unidad))
idx = []
for col, etq, color in [("kwh", "Consumo (índice)", COLOR_CONSUMO), ("fact", "Facturación (índice)", COLOR_FACTURA)]:
    ind = 100 * m[col] / base23[col].mean()
    fig.add_trace(go.Scatter(x=m.index, y=ind, mode="lines+markers", name=etq, line=dict(color=color), visible=False, connectgaps=False,
                             hovertemplate="%{x|%Y-%m}<br>%{y:.1f} (100 = promedio 2023)<extra></extra>"))
    idx.append(len(fig.data) - 1)
grupos.append(("Índice base 100 (promedio 2023): consumo y facturación", idx, "índice (2023 = 100)"))

sombrear_faltantes(fig)                      # añade franjas grises y una entrada de leyenda (última traza)
idx_leyenda = len(fig.data) - 1
botones = []
for nombre, idx, unidad in grupos:
    vis = [(i in idx) or (i == idx_leyenda) for i in range(len(fig.data))]
    botones.append(dict(label=nombre, method="update",
                        args=[{"visible": vis}, {"title.text": nombre, "yaxis.title.text": unidad}]))
fig.update_layout(updatemenus=[dict(buttons=botones, x=0.0, y=1.18, xanchor="left", direction="down")],
                  title=grupos[0][0], xaxis_title="Mes", yaxis_title=grupos[0][2], height=500,
                  legend=dict(orientation="h", y=-0.2))
fig.show()

## 8. ¿Por qué se mueve la facturación? Razón S/kWh y composición por tarifa

La facturación puede cambiar por **volumen** (kWh), por **precio/estructura tarifaria** o por **cargos fijos y ajustes**. Aquí descomponemos con una tabla mes × tarifa.

> **Hallazgo de calidad que se verifica en código:** algunas tarifas aparecen con `consumo_kwh = 0` en *todas* sus filas pero con facturación positiva. Para esas tarifas el kWh **no está informado** en estos archivos; incluirlas en una razón S/kWh la distorsionaría. Por eso la razón S/kWh se calcula **solo con tarifas que sí informan kWh**.

La razón S/kWh es una **razón económica observada**, no la tarifa contractual oficial.

In [18]:
cubo_pt = cachear("periodo_tarifa", """
SELECT periodo, tarifa, 'cliente' AS granularidad, COUNT(*)::BIGINT AS suministros_mes,
       SUM(consumo_kwh) AS kwh, SUM(facturacion_soles) AS fact,
       SUM(CASE WHEN consumo_kwh = 0 THEN 1 ELSE 0 END)::BIGINT AS n_kwh_cero
FROM clientes GROUP BY 1, 2
UNION ALL
SELECT periodo, tarifa, 'agregado', SUM(clientes)::BIGINT, SUM(consumo_kwh), SUM(facturacion_soles),
       SUM(CASE WHEN consumo_kwh = 0 THEN clientes ELSE 0 END)::BIGINT
FROM agregado GROUP BY 1, 2
""")
cubo_pt["fecha"] = pd.to_datetime(cubo_pt["periodo"].astype(str) + "01", format="%Y%m%d")

est_t = cubo_pt.groupby("tarifa").agg(suministros_mes=("suministros_mes", "sum"), kwh=("kwh", "sum"),
                                      fact=("fact", "sum"), n_kwh_cero=("n_kwh_cero", "sum"))
est_t["% filas con kWh = 0"] = 100 * est_t["n_kwh_cero"] / est_t["suministros_mes"]
est_t["% de la facturación"] = 100 * est_t["fact"] / est_t["fact"].sum()
TARIFAS_SIN_KWH = est_t.index[est_t["% filas con kWh = 0"] >= 99.0].tolist()
TARIFAS_CON_KWH = [t for t in est_t.index if t not in TARIFAS_SIN_KWH]
print("Tarifas SIN kWh informado (≥99 % de filas con kWh = 0):", TARIFAS_SIN_KWH)
print("Tarifas con kWh informado:", TARIFAS_CON_KWH)
display(est_t.sort_values("fact", ascending=False)[["suministros_mes", "kwh", "fact", "% filas con kWh = 0", "% de la facturación"]]
        .style.format({"suministros_mes": "{:,.0f}", "kwh": "{:,.0f}", "fact": "{:,.0f}", "% filas con kWh = 0": "{:.1f}", "% de la facturación": "{:.1f}"}))

# Control: el cubo debe reproducir el resumen
chk = cubo_pt.groupby("periodo")[["kwh", "fact"]].sum().join(resumen.set_index("periodo")[["consumo_kwh_total", "facturacion_soles_total"]])
assert np.allclose(chk["kwh"], chk["consumo_kwh_total"], rtol=1e-6) and np.allclose(chk["fact"], chk["facturacion_soles_total"], rtol=1e-6), "El cubo no coincide con el resumen"
print("El cubo periodo×tarifa reproduce los totales del resumen mensual.")

Tarifas SIN kWh informado (≥99 % de filas con kWh = 0): ['BT2', 'BT3', 'BT5A', 'MT2', 'MT3']
Tarifas con kWh informado: ['BT4', 'BT5B', 'BT5D', 'BT6', 'MT4']


,suministros_mes,kwh,fact,% filas con kWh = 0,% de la facturación
tarifa,,,,,
BT5B,"14,182,726","686,587,028","782,461,076",13.8,66.4
MT4,"25,326","133,785,531","169,012,302",11.5,14.4
MT2,"12,328",0,"110,519,130",100.0,9.4
MT3,"7,999",0,"70,569,475",100.0,6.0
BT5D,"80,925","57,423,657","29,818,453",0.8,2.5
BT4,"1,725","6,555,598","7,056,664",5.6,0.6
BT6,"12,138","3,588,603","4,491,665",4.1,0.4
BT3,658,0,"2,217,212",100.0,0.2
BT5A,563,0,"899,443",100.0,0.1


El cubo periodo×tarifa reproduce los totales del resumen mensual.


In [19]:
# --- Razón S/kWh solo con tarifas que informan kWh ---
ck = cubo_pt[cubo_pt["tarifa"].isin(TARIFAS_CON_KWH)]
razon_total = ck.groupby("fecha")[["fact", "kwh"]].sum()
razon_total["ratio"] = razon_total["fact"] / razon_total["kwh"]
razon_tar = ck.assign(ratio=ck["fact"] / ck["kwh"].replace(0, np.nan))
principales = est_t.loc[TARIFAS_CON_KWH].sort_values("fact", ascending=False).head(5).index.tolist()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Razón facturación/kWh (S/ por kWh) — solo tarifas con kWh informado",
                                    "Composición de la facturación: tarifas con kWh informado vs. sin kWh informado"))
fig.add_trace(go.Scatter(x=razon_total.index, y=razon_total["ratio"], mode="lines+markers", name="Total (tarifas con kWh)",
                         line=dict(color="black", width=3), connectgaps=False,
                         hovertemplate="%{x|%Y-%m}<br>%{y:.3f} S/kWh<extra>Total</extra>"), row=1, col=1)
for t in principales:
    g = razon_tar[razon_tar["tarifa"] == t].set_index("fecha")["ratio"].reindex(m.index)
    fig.add_trace(go.Scatter(x=g.index, y=g.values, mode="lines", name=t, connectgaps=False,
                             hovertemplate="%{x|%Y-%m}<br>%{y:.3f} S/kWh<extra>" + t + "</extra>"), row=1, col=1)
comp_fact = cubo_pt.assign(grupo=np.where(cubo_pt["tarifa"].isin(TARIFAS_SIN_KWH), "Tarifas sin kWh informado", "Tarifas con kWh informado"))
comp_fact = comp_fact.groupby(["fecha", "grupo"])["fact"].sum().unstack().reindex(m.index)
for gcol, color in [("Tarifas con kWh informado", "#1f77b4"), ("Tarifas sin kWh informado", "#9467bd")]:
    if gcol in comp_fact:
        fig.add_trace(go.Bar(x=comp_fact.index, y=comp_fact[gcol], name=gcol, marker_color=color,
                             hovertemplate="%{x|%Y-%m}<br>%{y:,.0f} S/<extra>" + gcol + "</extra>"), row=2, col=1)
fig.update_layout(barmode="stack", height=720, legend=dict(orientation="h", y=-0.12))
fig.update_yaxes(title_text="S/ por kWh", row=1, col=1); fig.update_yaxes(title_text="Facturación (S/)", row=2, col=1, tickformat=",.0f")
fig.update_xaxes(title_text="Mes", row=2, col=1)
fig.show()

# --- ¿Qué tarifas explican los mayores cambios mensuales de la facturación? ---
pv = cubo_pt.pivot_table(index="fecha", columns="tarifa", values="fact", aggfunc="sum").reindex(m.index)
delta = pv.diff()                                   # NaN si falta el mes anterior
delta_tot = m["fact"].diff()
top_cambios = delta_tot.dropna().abs().nlargest(4).index
contrib = (delta.loc[top_cambios].div(delta_tot.loc[top_cambios], axis=0) * 100).round(1)
contrib.insert(0, "Δ facturación total (S/)", delta_tot.loc[top_cambios].round(0))
contrib.index = contrib.index.strftime("%Y-%m")
print("\nContribución (%) de cada tarifa al cambio de facturación en los 4 mayores cambios mensuales:")
display(contrib.style.format("{:,.1f}"))


Contribución (%) de cada tarifa al cambio de facturación en los 4 mayores cambios mensuales:


tarifa,Δ facturación total (S/),BT2,BT3,BT4,BT5A,BT5B,BT5D,BT6,MT2,MT3,MT4
fecha,,,,,,,,,,,
2026-04,"5,861,187.0",0.0,0.0,0.5,0.0,36.4,-4.5,0.1,2.3,4.2,60.9
2026-08,"4,388,803.0",-0.0,0.1,-0.6,0.0,7.8,4.5,0.1,1.3,2.9,83.9
2025-11,"-4,381,469.0",0.0,0.0,0.0,-0.0,6.2,8.4,0.1,5.3,5.1,74.9
2026-07,"-3,709,304.0",0.0,0.0,-0.3,0.1,-5.2,6.3,0.0,4.7,2.8,91.5


## 9. Tendencia y estacionalidad (exploratorio)

Dos preguntas distintas:
1. **Tendencia:** ¿el consumo crece en el tiempo? Usamos la pendiente de una regresión sobre el logaritmo (crecimiento % anual) y el test no paramétrico de Kendall.
2. **Estacionalidad:** ¿hay un patrón repetido por mes calendario? Evaluamos (a) **consistencia entre años** de los residuos tras quitar una tendencia lineal y (b) una **descomposición STL** (opcional).

**Advertencias metodológicas.** Hay como máximo 3 ciclos anuales y 3 meses faltantes en 2023–2025; la STL requiere serie regular, así que **interpolamos linealmente** esos meses *solo* para este ejercicio exploratorio (se marcan en el gráfico). El análisis se hace con **kWh por suministro** para que el crecimiento de la cartera no se confunda con estacionalidad.

In [20]:
obs = m.loc[m["estado"] != "sin datos"].copy()
obs["t"] = np.arange(len(obs))                       # índice ordinal solo sobre meses observados
obs["t_cal"] = ((obs.index.year - obs.index.year.min()) * 12 + obs.index.month - 1)   # meses calendario transcurridos

# ---- Tendencia ----
def tendencia(serie, t, etiqueta):
    ok = serie.notna()
    lr = stats.linregress(t[ok], np.log(serie[ok]))
    tau, p = stats.kendalltau(t[ok], serie[ok])
    return {"serie": etiqueta, "meses": int(ok.sum()), "crecimiento anual estimado (%)": 100 * (np.exp(12 * lr.slope) - 1),
            "R² (log-lineal)": lr.rvalue ** 2, "tau de Kendall": tau, "p-valor tau": p}

det_obs = obs[obs["estado"] == "detalle"]
filas_t = []
for col, nombre in [("n", "Suministros"), ("kwh", "Consumo total"), ("kwh_ps", "kWh por suministro"), ("fact", "Facturación total")]:
    filas_t.append(tendencia(det_obs[col], det_obs["t_cal"], nombre + " (detalle 2023–2025)"))
    filas_t.append(tendencia(obs[col], obs["t_cal"], nombre + " (todos los meses, incluye 2026 agregado)"))
tabla_tend = pd.DataFrame(filas_t)
display(tabla_tend.style.format({"crecimiento anual estimado (%)": "{:+.2f}", "R² (log-lineal)": "{:.3f}", "tau de Kendall": "{:+.3f}", "p-valor tau": "{:.4f}"}))
print("Nota: con tendencia + estacionalidad mezcladas, estos p-valores son orientativos (las observaciones mensuales no son independientes).")

,serie,meses,crecimiento anual estimado (%),R² (log-lineal),tau de Kendall,p-valor tau
0,Suministros (detalle 2023–2025),33,+2.00,0.990,+0.985,0.0000
1,"Suministros (todos los meses, incluye 2026 agregado)",41,+2.16,0.969,+0.985,0.0000
2,Consumo total (detalle 2023–2025),33,+4.77,0.544,+0.644,0.0000
3,"Consumo total (todos los meses, incluye 2026 agregado)",41,+5.40,0.677,+0.707,0.0000
4,kWh por suministro (detalle 2023–2025),33,+2.72,0.291,+0.409,0.0006
5,"kWh por suministro (todos los meses, incluye 2026 agregado)",41,+3.17,0.450,+0.520,0.0000
6,Facturación total (detalle 2023–2025),33,+9.25,0.613,+0.610,0.0000
7,"Facturación total (todos los meses, incluye 2026 agregado)",41,+7.57,0.557,+0.583,0.0000


Nota: con tendencia + estacionalidad mezcladas, estos p-valores son orientativos (las observaciones mensuales no son independientes).


In [21]:
# ---- Consistencia del patrón por mes calendario entre años ----
ajuste = stats.linregress(obs["t_cal"], obs["kwh_ps"])
obs["tend_lineal"] = ajuste.intercept + ajuste.slope * obs["t_cal"]
obs["desvio_%"] = 100 * (obs["kwh_ps"] / obs["tend_lineal"] - 1)           # desvío respecto a la tendencia lineal

mapa = obs.pivot_table(index="anio", columns="mes", values="desvio_%").reindex(columns=range(1, 13))
perfil = obs.groupby("mes")["desvio_%"].agg(["mean", "std", "count"]).reindex(range(1, 13))
corr_anios = mapa.T.corr(method="spearman", min_periods=6)

fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=("Desvío vs. tendencia (%) por año y mes", "Perfil promedio por mes calendario"))
fig.add_trace(go.Heatmap(z=mapa.values, x=MESES, y=[str(a) for a in mapa.index], colorscale="RdBu_r", zmid=0,
                         colorbar=dict(title="% vs tendencia", x=0.5, len=0.8),
                         hovertemplate="%{y}-%{x}<br>%{z:+.1f}% vs. tendencia<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=MESES, y=perfil["mean"], error_y=dict(type="data", array=perfil["std"].fillna(0)),
                     marker_color=COLOR_CONSUMO, customdata=perfil["count"],
                     hovertemplate="%{x}<br>Desvío medio: %{y:+.2f}%<br>Años observados: %{customdata}<extra></extra>", showlegend=False), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1); fig.update_yaxes(title_text="Desvío medio (%)", row=1, col=2)
fig.update_layout(title="¿Se repite el patrón mensual entre años? (kWh por suministro, tras quitar tendencia lineal)", height=430)
fig.show()

print("Correlación de Spearman del perfil mensual entre años (mínimo 6 meses en común):")
display(corr_anios.style.format("{:.2f}"))
vals = corr_anios.values[np.triu_indices_from(corr_anios.values, k=1)]
corr_med = np.nanmedian(vals) if np.isfinite(vals).any() else np.nan
corr_min = np.nanmin(vals) if np.isfinite(vals).any() else np.nan
print(f"Correlación entre años: mediana = {corr_med:.2f} | mínima = {corr_min:.2f}")

Correlación de Spearman del perfil mensual entre años (mínimo 6 meses en común):


anio,2023,2024,2025,2026
anio,,,,
2023,1.00,0.93,0.84,0.90
2024,0.93,1.00,0.70,0.86
2025,0.84,0.70,1.00,0.89
2026,0.90,0.86,0.89,1.00


Correlación entre años: mediana = 0.88 | mínima = 0.70


In [22]:
# ---- STL (opcional): serie regular 2023-01..2025-12 con interpolación de los meses faltantes ----
fuerza_estacional = np.nan
if HAY_STATSMODELS:
    base = m.loc["2023-01-01":"2025-12-01", "kwh_ps"]
    interpolados = base.isna()
    serie = base.interpolate(method="time", limit_area="inside")
    if serie.isna().any() or len(serie) < 24:
        print("No hay serie suficiente para STL.")
    else:
        res = STL(serie, period=12, robust=True).fit()
        fuerza_estacional = max(0.0, 1 - res.resid.var() / (res.seasonal + res.resid).var())
        fuerza_tendencia  = max(0.0, 1 - res.resid.var() / (res.trend + res.resid).var())
        comps = [("Observado (kWh/suministro)", serie), ("Tendencia", res.trend), ("Estacional", res.seasonal), ("Residuo", res.resid)]
        fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=[c[0] for c in comps])
        for i, (nombre, s) in enumerate(comps, start=1):
            fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines+markers" if i == 1 else "lines", showlegend=False,
                                     line=dict(color=COLOR_DETALLE), hovertemplate="%{x|%Y-%m}<br>%{y:.3f}<extra>" + nombre + "</extra>"), row=i, col=1)
        fig.add_trace(go.Scatter(x=serie.index[interpolados], y=serie[interpolados], mode="markers", name="Mes interpolado (no observado)",
                                 marker=dict(color="red", size=10, symbol="diamond-open")), row=1, col=1)
        fig.update_layout(title=f"Descomposición STL (exploratoria) — fuerza estacional = {fuerza_estacional:.2f}, fuerza de tendencia = {fuerza_tendencia:.2f}",
                          height=760, legend=dict(orientation="h", y=-0.08))
        fig.show()
        print("Fuerza estacional (0 = nula, 1 = total): ", round(fuerza_estacional, 3))
        print("Meses interpolados:", [f.strftime("%Y-%m") for f in serie.index[interpolados]])
else:
    print("STL omitida (statsmodels no instalado).")

# ---- Veredicto prudente ----
if np.isfinite(fuerza_estacional) and np.isfinite(corr_min) and fuerza_estacional >= 0.6 and corr_min >= 0.5:
    VEREDICTO_ESTAC = "Hay INDICIOS CONSISTENTES de un patrón mensual repetido, pero con 3 años NO es evidencia consolidada."
elif np.isfinite(corr_med) and corr_med >= 0.3 and (not np.isfinite(fuerza_estacional) or fuerza_estacional >= 0.4):
    VEREDICTO_ESTAC = "Hay INDICIOS MODERADOS de patrón mensual; insuficientes para afirmar estacionalidad consolidada."
else:
    VEREDICTO_ESTAC = "NO hay evidencia suficiente de estacionalidad consolidada con estos datos."
print("\nVeredicto exploratorio:", VEREDICTO_ESTAC)

Fuerza estacional (0 = nula, 1 = total):  0.845
Meses interpolados: ['2024-06', '2024-12', '2025-02']

Veredicto exploratorio: Hay INDICIOS CONSISTENTES de un patrón mensual repetido, pero con 3 años NO es evidencia consolidada.


## 10. Hallazgos calculados

La siguiente celda **redacta los hallazgos a partir de los números reales** (nada está escrito a mano), de modo que si los datos cambian, el texto también.

In [23]:
def mes(f): return f.strftime("%Y-%m")
d = m[m["estado"] != "sin datos"]
H = []
H.append(f"COBERTURA: {len(d)} de {len(m)} meses del calendario tienen datos; sin datos: {[mes(f) for f in faltantes]}.")
H.append(f"CONSUMO: máximo en {mes(d['kwh'].idxmax())} ({fmt_n(d['kwh'].max())} kWh) y mínimo en {mes(d['kwh'].idxmin())} ({fmt_n(d['kwh'].min())} kWh).")
H.append(f"FACTURACIÓN: máximo en {mes(d['fact'].idxmax())} (S/ {fmt_n(d['fact'].max())}) y mínimo en {mes(d['fact'].idxmin())} (S/ {fmt_n(d['fact'].min())}).")
H.append(f"POR SUMINISTRO: máximo de kWh/suministro en {mes(d['kwh_ps'].idxmax())} ({d['kwh_ps'].max():.2f}) y mínimo en {mes(d['kwh_ps'].idxmin())} ({d['kwh_ps'].min():.2f}).")
H.append(f"CARTERA: de {fmt_n(d['n'].iloc[0])} suministros ({mes(d.index[0])}) a {fmt_n(d['n'].iloc[-1])} ({mes(d.index[-1])}); variación {100*(d['n'].iloc[-1]/d['n'].iloc[0]-1):+.1f}%.")
H.append(f"CAMBIOS INUSUALES (consumo): {[(mes(f), round(v,1)) for f,v in inus_kwh.items()]} | (facturación): {[(mes(f), round(v,1)) for f,v in inus_fac.items()]} (% mensual).")
if res_yoy["kwh_yoy_mediana"].notna().any():
    H.append("INTERANUAL (mediana %, por año observado): " + "; ".join(f"{int(a)}: consumo {r.kwh_yoy_mediana:+.1f}%, facturación {r.fact_yoy_mediana:+.1f}% ({int(r.meses)} meses)" for a, r in res_yoy.iterrows() if pd.notna(r.kwh_yoy_mediana)))
H.append(f"TARIFAS SIN kWh INFORMADO: {TARIFAS_SIN_KWH}, que aportan {est_t.loc[TARIFAS_SIN_KWH, 'fact'].sum()/est_t['fact'].sum()*100:.1f}% de la facturación acumulada.")
tt = tabla_tend.set_index("serie")
H.append(f"TENDENCIA (detalle 2023–2025): suministros {tt.loc['Suministros (detalle 2023–2025)','crecimiento anual estimado (%)']:+.2f}%/año; consumo total {tt.loc['Consumo total (detalle 2023–2025)','crecimiento anual estimado (%)']:+.2f}%/año; kWh por suministro {tt.loc['kWh por suministro (detalle 2023–2025)','crecimiento anual estimado (%)']:+.2f}%/año.")
H.append(f"ESTACIONALIDAD: {VEREDICTO_ESTAC}")
c0 = comp.set_index("indicador")["puntaje robusto del salto"]
H.append("CONTINUIDAD dic‑2025→ene‑2026 (puntaje robusto): " + ", ".join(f"{k} {v:+.1f}" for k, v in c0.items()) + " (|valor|>3 sería salto inusual).")
for h in H: print("•", h, "\n")

• COBERTURA: 41 de 44 meses del calendario tienen datos; sin datos: ['2024-06', '2024-12', '2025-02']. 

• CONSUMO: máximo en 2026-06 (25,139,146 kWh) y mínimo en 2023-03 (17,806,276 kWh). 

• FACTURACIÓN: máximo en 2026-08 (S/ 34,968,838) y mínimo en 2023-03 (S/ 21,899,917). 

• POR SUMINISTRO: máximo de kWh/suministro en 2026-06 (68.89) y mínimo en 2023-03 (52.84). 

• CARTERA: de 336,736 suministros (2023-01) a 366,818 (2026-08); variación +8.9%. 

• CAMBIOS INUSUALES (consumo): [('2023-04', 15.4), ('2025-04', 8.5), ('2026-04', 11.2)] | (facturación): [] (% mensual). 

• INTERANUAL (mediana %, por año observado): 2024: consumo +3.5%, facturación +18.4% (10 meses); 2025: consumo +3.6%, facturación +4.6% (9 meses); 2026: consumo +9.0%, facturación +1.7% (7 meses) 

• TARIFAS SIN kWh INFORMADO: ['BT2', 'BT3', 'BT5A', 'MT2', 'MT3'], que aportan 15.7% de la facturación acumulada. 

• TENDENCIA (detalle 2023–2025): suministros +2.00%/año; consumo total +4.77%/año; kWh por suministro +2.72

## 11. Conclusiones

Las cifras exactas están en la celda anterior (se calculan al ejecutar). Aquí se organiza la lectura:

**1. ¿Qué descubrimos?** — Revise los puntos *COBERTURA*, *CONSUMO*, *FACTURACIÓN*, *CARTERA* y *CAMBIOS INUSUALES*. Distinga siempre entre (a) meses sin archivo, (b) cambios de volumen (kWh) y (c) cambios de facturación sin cambio de kWh.

**2. ¿Qué indicadores y gráficos lo respaldan?** — Mapa de cobertura (sección 2), series mensuales (4), comparación por mes calendario (5), variaciones MoM/YoY (6), selector de variable (7), razón S/kWh y contribuciones por tarifa (8), tendencia y estacionalidad (9).

**3. ¿Por qué importa para Electro Puno?** — La demanda por suministro permite proyectar energía a comprar y cargas de red; la facturación que se mueve sin kWh apunta a efectos de precio, cargos o ajustes que deben gestionarse comercialmente; los huecos de datos impiden cerrar series anuales.

**4. Acciones o análisis adicionales recomendados**
* Recuperar los **archivos mensuales faltantes** (o confirmar que no existen) antes de reportar anualizados.
* Contrastar los meses con **cambios inusuales de facturación** con el calendario de reajustes tarifarios y regularizaciones.
* Pedir a quien administra la fuente la **medición de kWh para las tarifas con kWh = 0** (necesaria para estimar demanda total real).
* Repetir el análisis de estacionalidad cuando haya ≥ 5 años completos, o con datos diarios/horarios si existen.

**5. Limitaciones**
* 2026 es agregado: sirve para totales y suministros, no para distribuciones individuales.
* Máximo 3 ciclos anuales; interpolación solo en el ejercicio STL.
* El consumo total **subestima** la energía real porque algunas tarifas no informan kWh.
* La razón S/kWh es observada y no equivale a la tarifa oficial.
* Los cambios mensuales "inusuales" son estadísticos; no se interpretan como error ni irregularidad sin más información.